وقتی داده ها فیلتر شدند این برنامه نمودار تولید بر اساس دما را برای تمام واحدهای تولیدی رسم کرده و داده های مربوط به آخرین سری تولید و داده های انتخاب شده توسط فیلتر پنجم را نیز مشخص میکند. 

In [36]:
import os
import sys
import plotly.graph_objects as go

import matplotlib.pyplot as plt
from sklearn.linear_model import RANSACRegressor, LinearRegression
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
from src.models.filter_data.filter_data import *
from src.models.filter_data.feature_adder import *

In [37]:
l_min = 4
max_diff = 3
c_thresh = 0.9

csv_read_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")

df = pd.read_csv(csv_read_path, encoding='utf-8')
df_c = df

In [38]:
csv_read_path = os.path.join(project_root, "data", "interim", "factors.csv")
df_factors = pd.read_csv(csv_read_path)

coefs = {}
grouped = df_factors.groupby(['PowerPlantCode', 'PowerPlantName', "UnitCode"])

for (pp_code, pp_name, unit_code), g in grouped:
    uniques = g[["a1IndexGas", "b1IndexGas"]].drop_duplicates()
    coefs[(pp_name, unit_code)] = []
    for row in uniques.itertuples(index=False):
        coefs[(pp_name, unit_code)].append((row.a1IndexGas, row.b1IndexGas))

In [39]:
def select_envelope_neighbors_indices(X, y, one_unit_df, temp_feature, alpha, beta):
    # model = LinearRegression()#PolynomialModel(degree=2)
    # model.fit(X, y)
    
    model = RANSACRegressor(estimator=LinearRegression(), min_samples=0.9)

    sens_temps = one_unit_df[temp_feature].values
    gens = one_unit_df['generation'].values
    
    model.fit(X, y)
    X_all = sens_temps.reshape(-1, 1)
    y_pred_all = model.predict(X_all)
    is_in_area = (y_pred_all + alpha >= gens) & (gens >= y_pred_all - beta)
    return model

In [40]:
import plotly.express as px


def show(df_m1, save=False, param=None, ass=""):
    temp_feature = 'temp_sens'
    features = ["name", "code", "generation", f"{temp_feature}", "is_good_peak"]
    df_modified = df_m1[features].copy(deep=True)
    df_modified = df_modified[df_modified["is_good_peak"] >= 5]
    ds = Data_selector(df_modified)

    name, code = param['name'], param['code']
    one_unit_df = ds.filter_name_code(name, code)

    sens_temps = one_unit_df[temp_feature].values
    gens = one_unit_df['generation'].values

    X, y = find_points_on_envelope(gens, sens_temps, bin_length=100, p=0.98)

    model = select_envelope_neighbors_indices(X, y, one_unit_df,temp_feature, alpha=2,beta=5)
    y_pred_model = model.predict(X)
    
    fig = px.scatter(df_m1, x="datetime", y='generation', color='is_good_peak',
                     title='Generation over Time by Batch Interval',
                     labels={'generation': 'Generation', 'datetime': 'Time'},
                     hover_data=['datetime', 'generation', 'temp_sens'])

    if save:
        fig.write_html(
            f"{project_root}/src/visualization/unit_figs/filter5{ass}/{param['name']}-{param['code']}_l.html")
    else:
        fig.show()

    fig = px.scatter(df_m1, x="temp_sens", y='generation', color='is_good_peak',
                     title='Generation over Time by Batch Interval',
                     labels={'generation': 'Generation'},
                     hover_data=['datetime', 'generation', 'temp_sens'],size_max=1)
    
    fig.update_traces(
    marker=dict(
        size=4,           # اندازه ثابت
        sizemode='diameter',  # یا 'area'
        sizeref=1,        # برای مقیاس‌بندی
        opacity=0.7       # شفافیت
    )
    )
    
    n = param["name"]
    c = param["code"]
    for i in range(len(coefs[(n, c)])):
        a, b = coefs[(n, c)][i]
        x_line = np.linspace(df_m1["temp_sens"].min(), df_m1["temp_sens"].max(), 100)
        y_line = a * x_line + b
        
        fig.add_trace(go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            name=f"y = {a}x + {b}",
            line=dict(dash="dash", width=2)
        ))
    
    df_points = pd.DataFrame({
    'sens_temp': X.flatten(),
    'generation_env': y,
    "y_pred_model" : y_pred_model
    })
    
    # اضافه کردن نقاط envelope (به صورت scatter دوم)
    fig.add_scatter(
        x=df_points['sens_temp'], 
        y=df_points['generation_env'], 
        mode='markers', 
        marker=dict(size=10, color='red'),
        name='Envelope Points'
    )
    
    fig.add_trace(go.Scatter(
            x=df_points['sens_temp'],
            y=df_points['y_pred_model'],
            mode="lines",
            line=dict(dash="solid", width=2)
        ))
    
    
    fig.add_trace(go.Scatter(
            x=df_points['sens_temp'],
            y=df_points['y_pred_model']-5,
            mode="lines",
            line=dict(dash="solid", width=2)
        ))
    
    
    fig.add_trace(go.Scatter(
            x=df_points['sens_temp'],
            y=df_points['y_pred_model']+2,
            mode="lines",
            line=dict(dash="solid", width=2)
        ))
    
    if save:
        fig.write_html(
            f"{project_root}/src/visualization/unit_figs/filter5{ass}/{param['name']}-{param['code']}_s.html")
    else:
        fig.show()
        
    

In [41]:
power_plants = df_c[['name', 'code']].drop_duplicates()

for row in power_plants.itertuples():
    name_plot, code_plot = row.name, row.code
    ds_n_c_plot = Data_selector(Data_selector(df_c).select_peaks(goodness=2))
    df_n_c_plot = ds_n_c_plot.filter_name_code(name_plot, code_plot)
    try:
        show(df_n_c_plot, save=True, param={"name": name_plot, "code": code_plot})
    except:
        print(name_plot,code_plot)

U:\ML_project\Bargh_ML\venv\Lib\site-packages\sklearn\metrics\_regression.py:1283: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

U:\ML_project\Bargh_ML\venv\Lib\site-packages\sklearn\metrics\_regression.py:1283: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

U:\ML_project\Bargh_ML\venv\Lib\site-packages\sklearn\metrics\_regression.py:1283: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

U:\ML_project\Bargh_ML\venv\Lib\site-packages\sklearn\metrics\_regression.py:1283: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

U:\ML_project\Bargh_ML\venv\Lib\site-packages\sklearn\metrics\_regression.py:1283: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.

U:\ML_project\Bargh_ML\venv\Lib\site-packages\sklearn\metrics\_regression.py:1283: UndefinedMetricWarning:

R^2 score is not well-defined with less than two samples.